# 📱 Google Play Store Dashboard

Evaluación 3 Duoc UC

## Paso 1: Instalar

In [1]:
!pip install -q dash plotly pandas

## Paso 2: Datos

In [2]:
import pandas as pd, numpy as np

df_raw = pd.read_csv('./data/raw/googleplaystore.csv')
print(f"✓ {df_raw.shape}") 

✓ (10841, 13)


## Paso 3: Limpiar

In [3]:
df = df_raw.copy()
df = df.drop_duplicates(subset=["App"])
df["Rating"] = df["Rating"].fillna(df["Rating"].median())
df["Installs"] = df["Installs"].str.replace("+","").str.replace(",","")
df["Installs"] = pd.to_numeric(df["Installs"], errors="coerce").fillna(0)
df["Reviews"] = pd.to_numeric(df["Reviews"], errors="coerce").fillna(0)
df["Price"] = df["Price"].str.replace("$","")
df["Price"] = pd.to_numeric(df["Price"], errors="coerce").fillna(0)
df = df.dropna(subset=["App","Category","Type"])
df = df[df["Installs"] > 0]
df = df[(df["Rating"] >= 0) & (df["Rating"] <= 5)]

print(f"✓ Limpio: {df.shape}")
print(f"Apps: {len(df)}, Pagadas: {(df['Type']=='Paid').mean()*100:.1f}%, Rating: {df['Rating'].mean():.2f}")

✓ Limpio: (9644, 13)
Apps: 9644, Pagadas: 7.7%, Rating: 4.19


## Paso 4: Dashboard

In [4]:
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import dash_bootstrap_components as dbc 

# 1. Inicializar la app usando el tema FLATLY de Bootstrap
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.FLATLY])

# 2. Construir la Barra Lateral (Sidebar) para los filtros
sidebar = html.Div(
    [
        html.H3("Filtros", className="display-6"),
        html.Hr(),
        html.P("Ajuste los parámetros del análisis:", className="lead"),
        
        dbc.Label("Categoría:"),
        dcc.Dropdown(
            id='filtro_categoria', 
            options=[{'label': 'Todas las Categorías', 'value': 'ALL'}] + [{'label': c, 'value': c} for c in sorted(df['Category'].unique())], 
            value='ALL', clearable=False, className="mb-4"
        ),
        
        dbc.Label("Público Objetivo (Content Rating):"),
        dcc.Dropdown(
            id='filtro_content', 
            options=[{'label': 'Todos los Públicos', 'value': 'ALL'}] + [{'label': cr, 'value': cr} for cr in df['Content Rating'].dropna().unique()], 
            value='ALL', clearable=False, className="mb-4"
        )
    ],
    style={"padding": "2rem", "backgroundColor": "#f8f9fa", "minHeight": "100vh"}
)

# 3. Construir el Contenido Principal (Gráficos en Tarjetas)
content = html.Div(
    [
        html.H1("Google Play Store Analytics", className="mb-4 mt-2"),
        
        # Fila 1: Visión General
        dbc.Row([
            dbc.Col(dbc.Card(dbc.CardBody([dcc.Graph(id='grafico_tipo')])), md=4),
            dbc.Col(dbc.Card(dbc.CardBody([dcc.Graph(id='grafico_rating_dist')])), md=8),
        ], className="mb-4"),
        
        # Fila 2: Categorías y Públicos
        dbc.Row([
            dbc.Col(dbc.Card(dbc.CardBody([dcc.Graph(id='grafico_categorias')])), md=6),
            dbc.Col(dbc.Card(dbc.CardBody([dcc.Graph(id='grafico_content')])), md=6),
        ], className="mb-4"),
        
        # Fila 3: Monetización
        dbc.Row([
            dbc.Col(dbc.Card(dbc.CardBody([dcc.Graph(id='grafico_precio')])), md=12),
        ], className="mb-4"),
    ],
    style={"padding": "2rem"}
)

# 4. Unir Sidebar y Contenido en el Layout principal
app.layout = dbc.Container(
    [
        dbc.Row(
            [
                dbc.Col(sidebar, md=3, style={"padding": "0px"}),
                dbc.Col(content, md=9)
            ],
            className="g-0" 
        )
    ],
    fluid=True,
    style={"margin": "0px", "padding": "0px"}
)

# 5. Callbacks 
@app.callback(
    Output('grafico_tipo', 'figure'),
    Output('grafico_rating_dist', 'figure'),
    Output('grafico_content', 'figure'),
    Output('grafico_categorias', 'figure'),
    Output('grafico_precio', 'figure'),
    Input('filtro_categoria', 'value'),
    Input('filtro_content', 'value')
)
def actualizar_graficos(cat_sel, content_sel):
    dff = df.copy()
    if cat_sel != 'ALL': dff = dff[dff['Category'] == cat_sel]
    if content_sel != 'ALL': dff = dff[dff['Content Rating'] == content_sel]

    tema = 'plotly_white'
    margen = dict(l=20, r=20, t=40, b=20)

    # 1. Free vs Paid
    fig_tipo = px.pie(dff, names='Type', title='Free vs Paid', hole=0.4, color_discrete_sequence=['#2ECC71', '#3498DB'])
    fig_tipo.update_layout(template=tema, margin=margen)

    # 2. Rating Dist
    fig_rating = px.histogram(dff, x='Rating', nbins=20, title='Distribución de Calificaciones', color_discrete_sequence=['#9B59B6'])
    fig_rating.update_layout(template=tema, margin=margen, yaxis_title="Apps", xaxis_title="Rating")

    # 3. Content Rating
    cr_counts = dff['Content Rating'].value_counts().reset_index()
    cr_counts.columns = ['Público', 'Apps']
    fig_content = px.bar(cr_counts, x='Público', y='Apps', title='Apps por Público', color_discrete_sequence=['#F1C40F'])
    fig_content.update_layout(template=tema, margin=margen, yaxis_tickformat='~s')

    # 4. Top Categorías
    top_cats = dff['Category'].value_counts().head(10).reset_index()
    top_cats.columns = ['Categoría', 'Apps']
    fig_cats = px.bar(top_cats, y='Categoría', x='Apps', orientation='h', title='Top 10 Categorías', color_discrete_sequence=['#E74C3C'])
    fig_cats.update_layout(template=tema, margin=margen, xaxis_tickformat='~s', yaxis={'categoryorder':'total ascending'})

    # 5. Precios
    dff_paid = dff[dff['Type'] == 'Paid']
    if dff_paid.empty:
        fig_precio = px.box(title='Distribución de Precios (Sin Datos)')
    else:
        fig_precio = px.box(dff_paid, x='Price', title='Distribución de Precios (Apps de Pago)', color_discrete_sequence=['#E67E22'])
    fig_precio.update_layout(template=tema, margin=margen)

    return fig_tipo, fig_rating, fig_content, fig_cats, fig_precio

# app.run_server(debug=True)

## Paso 5: Ejecutar

http://127.0.0.1:8050/

In [ ]:
import threading, time

t = threading.Thread(target=lambda: app.run(mode='external', port=8050, debug=False, host='127.0.0.1'), daemon=True)
t.start()
time.sleep(3)

print("✅ DASHBOARD ACTIVO")
print(f"Total apps: {len(df)}")
print(f"Rating: {df['Rating'].mean():.2f}/5.0")

try:
    while True: time.sleep(1)
except: pass

✅ DASHBOARD ACTIVO
Total apps: 9644
Rating: 4.19/5.0
